In [15]:
import pandas as pd
import requests 
from shapely.geometry import Point
import geopandas as gpd
import numpy as np
from scipy.spatial import cKDTree
import re 
import unicodedata
from Bio import Phylo
import xarray as xr

# Language Data 

In [16]:
iso3_extra_map = {
        'en': 'eng',
    'fr': 'fra',
    'zh': 'cmn',
    'zho':'cmn',
    'es': 'spa',
    'ru': 'rus',
    'de': 'deu',
    'ja': 'jpn',
    'pt': 'por',
    'ko': 'kor',
    'ar': 'arb',
    'ara':'arb',
    'it': 'ita',
    'vi': 'vie',
    'tr': 'tur',
    'hi': 'hin',
    'id': 'ind',
    'pl': 'pol',
    'nl': 'nld',
    'th': 'tha',
    'cs': 'ces',
    'bn': 'ben',
    'fa': 'fas',
    'ro': 'ron',
    'sv': 'swe',
    'fi': 'fin',
    'da': 'dan',
    'el': 'ell',
    'ta': 'tam',
    'ca': 'cat',
    'uk': 'ukr',
    'hu': 'hun',
    'te': 'tgl',
    'bg': 'bul',
    'ur': 'urd',
    'sl': 'slv',
    'ml': 'mal',
    'he': 'heb',
    'et': 'est',
    'sk': 'slk',
    'ms': 'msa',
    'mr': 'mar',
    'sw': 'swa',
    'lt': 'lit',
    'cy': 'cym',
    'eu': 'eus',
    'hr': 'hrv',
    'gu': 'guj',
    'kn': 'kan',
    'sr': 'srp',
    'no': 'nor',
    'lv': 'lav',
    'pa': 'pan',
    'az': 'aze',
    'am': 'amh',
    'is': 'isl',
    'yo': 'yor',
    'ga': 'gle',
    'kk': 'kaz',
    'gl': 'glg',
    'ne': 'nep',
    'mt': 'mlt',
    'si': 'sin',
    'af': 'afr',
    'or': 'ori',
    'sq': 'sqi',
    'ha': 'hau',
    'uz': 'uzb',
    'ka': 'kat',
    'mk': 'mkd',
    'tl': 'tgl',
    'as': 'asm',
    'my': 'mya',
    'be': 'bel',
    'hy': 'hye',
    'ig': 'ibo',
    'eo': 'epo',
    'br': 'bre',
    'km': 'khm',
    'la': 'lat',
    'mn': 'mon',
    'ky': 'kir',
    'nb': 'nob',
    'zu': 'zul',
    'so': 'som',
    'jv': 'jav',
    'rw': 'kin',
    'xh': 'xho',
    'bs': 'bos',
    'ps': 'pus',
    'nn': 'nno',
    'wo': 'wol',
    'ku': 'kur',
    'tt': 'tat',
    'sa': 'san',
    'lo': 'lao',
    'lg': 'lug',
    'ug': 'uig',
    'tw': 'twi',
    'gd': 'gla',
    'tg': 'tgk',
    'sd': 'snd',
    'lb': 'ltz',
    'su': 'sun',
    'gn': 'grn',
    'tk': 'tuk',
    'oc': 'oci',
    'ht': 'hat',
    'fy': 'fry',
    'sn': 'sna',
    'bo': 'bod',
    'yi': 'yid',
    'ba': 'bak',
    'ln': 'lin',
    'bm': 'bam',
    'fo': 'fao',
    'dv': 'div',
    'tn': 'tsn',
    'ti': 'tir',
    'mg': 'mlg',
    'mi': 'mri',
    'ny': 'nya',
    'aa': 'aar',
    'cv': 'chv',
    'ab': 'abk',
    'st': 'sot',
    'ee': 'ewe',
    'ia': 'ina',
    'sc': 'srd',
    'qu': 'que',
    'li': 'lim',
    'om': 'orm',
    'ts': 'tso',
    'ak': 'aka',
    'rn': 'run',
    'rm': 'roh',
    'ch': 'cha',
    'sm': 'smo',
    'ks': 'kas',
    'ki': 'kik',
    'ss': 'ssw',
    'an': 'arg',
    'io': 'ido',
    'kg': 'kon',
    'ce': 'che',
    'dz': 'dzo',
    'os': 'oss',
    'wa': 'wln',
    'co': 'cos',
    'vo': 'vol',
    'fj': 'fij',
    'kw': 'cor',
    'sg': 'sag',
    'ay': 'aym',
    'bh': 'bih',
    'to': 'ton',
    'ie': 'ile',
    've': 'ven',
    'gv': 'glv',
    'av': 'ava',
    'kv': 'kom',
    'se': 'sme',
    'ff': 'ful',
    'nv': 'nav',
    'bi': 'bis',
    'kl': 'kal',
    'iu': 'iku',
    'ty': 'tah',
    'cu': 'chu',
    'za': 'zha',
    'ae': 'ave',
    'mh': 'mah',
    'pi': 'pli',
    'cr': 'cre',
    'ik': 'ipk',
    'ho': 'hmo',
    'kr': 'kau',
    'ng': 'pcm',
    'ii': 'iii',
    'kj': 'kua',
    'na': 'nau',
    'nr': 'nbl',
    'nd': 'nde',
    'oj': 'oji',
    'hz': 'her',
    'lu': 'lub',
    'hy_am': 'hye',
 'nl_NL': 'nld',
 'zh_cn': 'cmn',
 'es_CL': 'spa',
 'pt_PT': 'por',
 'es_DO': 'spa',
 'ar_TN': 'ara',
 'hi_in': 'hin',
 'es_cl': 'spa',
 'en_gb': 'eng',
 'cbk_zam': 'cbk',
 'zh_tw': 'cmn',
 'es_AR': 'spa',
 'es_ES': 'spa',
 'tl_PH': 'tgl',
 'ar_SY': 'arb',
 'rm_vallader': 'roh',
 'gu_in': 'guj',
 'vi_VN': 'vie',
 'nl_BE': 'nld',
 'zh_yue': 'cmn',
 'zh_hk': 'cmn',
 'bg_BG': 'bul',
 'zh_TW': 'cmn',
 'nb_no': 'nor',
 'es_PE': 'spa',
 'zh_HK': 'cmn',
 'es_mx': 'spa',
 'pa_in': 'pan',
 'tr_TR': 'tur',
 'en_CA': 'eng',
 'de_DE': 'deu',
 'ne_np': 'nep',
 'es_NI': 'spa',
 'de_AT': 'deu',
 'es_HN': 'spa',
 'bn_IN': 'ben',
 'es_MX': 'spa',
 'map_bms': 'map',
 'es_GT': 'spa',
 'es_EC': 'spa',
 'bn_bd': 'ben',
 'tg_TJ': 'tgk',
 'nb_NO': 'nor',
 'fa_AF': 'fas',
 'pt_br': 'por',
 'da_DK': 'dan',
 'es_SV': 'spa',
 'hi_IN': 'hin',
 'en_AU': 'eng',
 'ta_LK': 'tam',
 'en_ZA': 'eng',
 'roa_tara': 'roa',
 'no_nb': 'nor',
 'ur_PK': 'urdu',
 'nn_NO': 'nor',
 'en_GB': 'eng',
 'es_PR': 'spa',
 'en_NZ': 'eng',
 'es_PA': 'spa',
 'nn_no': 'nor',
 'es_VE': 'spa',
 'be_tarask': 'bel',
 'en_US': 'eng',
 'fr_FR': 'fra',
 'en_ca': 'eng',
 'es_CR': 'spa',
 'en_za': 'eng',
 'sr_ME': 'srp',
 'it_IT': 'ita',
 'de_CH': 'deu',
 'fr_BE': 'fra',
 'es_ar': 'spa',
 'rm_sursilv': 'roh',
 'fr_CA': 'fra',
 'zh_CN': 'cmn',
 'sv_se': 'swe',
 'ms_MY': 'msa',
 'fa_IR': 'fas',
 'az_IR': 'aze',
 'fr_ca': 'fra',
 'pt_BR': 'por',
 'es_CO': 'spa',
 'ga_ie': 'gle',
 'ca_valencia': 'cat',
 'es_UY': 'spa'}


iso_3_map = {
    'en': 'eng',
    'fr': 'fra',
    'zh': 'zho',
    'es': 'spa',
    'ru': 'rus',
    'de': 'deu',
    'ja': 'jpn',
    'pt': 'por',
    'ko': 'kor',
    'ar': 'ara',
    'it': 'ita',
    'vi': 'vie',
    'tr': 'tur',
    'hi': 'hin',
    'id': 'ind',
    'pl': 'pol',
    'nl': 'nld',
    'th': 'tha',
    'cs': 'ces',
    'bn': 'ben',
    'fa': 'fas',
    'ro': 'ron',
    'sv': 'swe',
    'fi': 'fin',
    'da': 'dan',
    'el': 'ell',
    'ta': 'tam',
    'ca': 'cat',
    'uk': 'ukr',
    'hu': 'hun',
    'te': 'tgl',
    'bg': 'bul',
    'ur': 'urd',
    'sl': 'slv',
    'ml': 'mal',
    'he': 'heb',
    'et': 'est',
    'sk': 'slk',
    'ms': 'msa',
    'mr': 'mar',
    'sw': 'swa',
    'lt': 'lit',
    'cy': 'cym',
    'eu': 'eus',
    'hr': 'hrv',
    'gu': 'guj',
    'kn': 'kan',
    'sr': 'srp',
    'no': 'nor',
    'lv': 'lav',
    'pa': 'pan',
    'az': 'aze',
    'am': 'amh',
    'is': 'isl',
    'yo': 'yor',
    'ga': 'gle',
    'kk': 'kaz',
    'gl': 'glg',
    'ne': 'nep',
    'mt': 'mlt',
    'si': 'sin',
    'af': 'afr',
    'or': 'ori',
    'sq': 'sqi',
    'ha': 'hau',
    'uz': 'uzb',
    'ka': 'kat',
    'mk': 'mkd',
    'tl': 'tgl',
    'as': 'asm',
    'my': 'mya',
    'be': 'bel',
    'hy': 'hye',
    'ig': 'ibo',
    'eo': 'epo',
    'br': 'bre',
    'km': 'khm',
    'la': 'lat',
    'mn': 'mon',
    'ky': 'kir',
    'nb': 'nob',
    'zu': 'zul',
    'so': 'som',
    'jv': 'jav',
    'rw': 'kin',
    'xh': 'xho',
    'bs': 'bos',
    'ps': 'pus',
    'nn': 'nno',
    'wo': 'wol',
    'ku': 'kur',
    'tt': 'tat',
    'sa': 'san',
    'lo': 'lao',
    'lg': 'lug',
    'ug': 'uig',
    'tw': 'twi',
    'gd': 'gla',
    'tg': 'tgk',
    'sd': 'snd',
    'lb': 'ltz',
    'su': 'sun',
    'gn': 'grn',
    'tk': 'tuk',
    'oc': 'oci',
    'ht': 'hat',
    'fy': 'fry',
    'sn': 'sna',
    'bo': 'bod',
    'yi': 'yid',
    'ba': 'bak',
    'ln': 'lin',
    'bm': 'bam',
    'fo': 'fao',
    'dv': 'div',
    'tn': 'tsn',
    'ti': 'tir',
    'mg': 'mlg',
    'mi': 'mri',
    'ny': 'nya',
    'aa': 'aar',
    'cv': 'chv',
    'ab': 'abk',
    'st': 'sot',
    'ee': 'ewe',
    'ia': 'ina',
    'sc': 'srd',
    'qu': 'que',
    'li': 'lim',
    'om': 'orm',
    'ts': 'tso',
    'ak': 'aka',
    'rn': 'run',
    'rm': 'roh',
    'ch': 'cha',
    'sm': 'smo',
    'ks': 'kas',
    'ki': 'kik',
    'ss': 'ssw',
    'an': 'arg',
    'io': 'ido',
    'kg': 'kon',
    'ce': 'che',
    'dz': 'dzo',
    'os': 'oss',
    'wa': 'wln',
    'co': 'cos',
    'vo': 'vol',
    'fj': 'fij',
    'kw': 'cor',
    'sg': 'sag',
    'ay': 'aym',
    'bh': 'bih',
    'to': 'ton',
    'ie': 'ile',
    've': 'ven',
    'gv': 'glv',
    'av': 'ava',
    'kv': 'kom',
    'se': 'sme',
    'ff': 'ful',
    'nv': 'nav',
    'bi': 'bis',
    'kl': 'kal',
    'iu': 'iku',
    'ty': 'tah',
    'cu': 'chu',
    'za': 'zha',
    'ae': 'ave',
    'mh': 'mah',
    'pi': 'pli',
    'cr': 'cre',
    'ik': 'ipk',
    'ho': 'hmo',
    'kr': 'kau',
    'ng': 'pcm',
    'ii': 'iii',
    'kj': 'kua',
    'na': 'nau',
    'nr': 'nbl',
    'nd': 'nde',
    'oj': 'oji',
    'hz': 'her',
    'lu': 'lub'
}

n_speakers_map = {
    '0': 0,
    '1 million': 1000000,
    '1,000,000': 1000000,
    '1,042,989': 1042989,
    '1,546,000': 1546000,
    '1,841,963': 1841963,
    '1,900': 1900,
    '1,973,800': 1973800,
    '1.0 million': 1000000,
    '1.1 million': 1100000,
    '1.2 billion': 1200000000,
    '1.2 million': 1200000,
    '1.3 million': 1300000,
    '1.334 million': 1334000,
    '1.4 million': 1400000,
    '1.4–3.5 million': (1400000 + 3500000) / 2,  # Average of the range
    '1.5 million': 1500000,
    '1.7 million': 1700000,
    '1.75 million': 1750000,
    '1.76 million': 1760000,
    '10 million': 10000000,
    '10,000': 10000,
    '10,000–12,000': (10000 + 12000) / 2,  # Average of the range
    '10.7 million': 10700000,
    '100,000 to 800,000': (100000 + 800000) / 2,  # Average of the range
    '1000': 1000,
    '108,000': 108000,
    '11 million': 11000000,
    '11,244,750': 11244750,
    '113 million': 113000000,
    '12 million': 12000000,
    '120,000': 120000,
    '13 million': 13000000,
    '13.2 million': 13200000,
    '13.5 million': 13500000,
    '130,000': 130000,
    '14,000,000': 14000000,
    '147,000': 147000,
    '148,720': 148720,
    '15 million': 15000000,
    '15 to 20 million': (15000000 + 20000000) / 2,  # Average of the range
    '15,000': 15000,
    '15,311,351': 15311351,
    '15,512,000': 15512000,
    '150 million': 150000000,
    '150,000': 150000,
    '150,000–200,000': (150000 + 200000) / 2,  # Average of the range
    '15000000': 15000000,
    '152,050': 152050,
    '1520 to ~2100': (1520 + 2100) / 2,  # Average of the range
    '16 million': 16000000,
    '160,000': 160000,
    '1689000': 1689000,
    '17 million': 17000000,
    '170,000': 170000,
    '171,080': 171080,
    '187,000': 187000,
    '190,110': 190110,
    '2 million': 2000000,
    '2 million to 18 million': (2000000 + 18000000) / 2,  # Average of the range
    '2,000': 2000,
    '2,000,000': 2000000,
    '2,144': 2144,
    '2.3 million': 2300000,
    '2.320 million': 2320000,
    '2.4 million': 2400000,
    '2.5 million': 2500000,
    '20,000': 20000,
    '200,000': 200000,
    '20–36 million': (20000000 + 36000000) / 2,  # Average of the range
    '21 million': 21000000,
    '21,807,730': 21807730,
    '210,000': 210000,
    '211,700': 211700,
    '22 million': 22000000,
    '22.5 million': 22500000,
    '220,000': 220000,
    '227,000': 227000,
    '23 million': 23000000,
    '23.6–24 million': (23600000 + 24000000) / 2,  # Average of the range
    '230 million': 230000000,
    '25 million': 25000000,
    '250 million': 250000000,
    '250,000': 250000,
    '250,000–300,000': (250000 + 300000) / 2,  # Average of the range
    '26 million': 26000000,
    '27.5 million': 27500000,
    '280,000': 280000,
    '3,160,000': 3160000,
    '3,600': 3600,
    '3.0 million': 3000000,
    '3.2 million': 3200000,
    '3.3 million': 3300000,
    '3.5 million': 3500000,
    '3.6 million': 3600000,
    '3.7 million': 3700000,
    '3.8 million': 3800000,
    '3.9 million': 3900000,
    '30 million': 30000000,
    '30,000': 30000,
    '30.551 million': 30551000,
    '31,000': 31000,
    '310,000': 310000,
    '314,000': 314000,
    '32,000,000': 32000000,
    '322 million': 322000000,
    '324,000': 324000,
    '33 million': 33000000,
    '33.9 million': 33900000,
    '330000': 330000,
    '339,210': 339210,
    '34 million': 34000000,
    '340,000': 340000,
    '341,300': 341300,
    '344,000': 344000,
    '35 million': 35000000,
    '350 million': 350000000,
    '350,000': 350000,
    '351,791': 351791,
    '360–400 million': (360000000 + 400000000) / 2,  # Average of the range
    '37,400,000': 37400000,
    '39,475': 39475,
    '390,000': 390000,
    '4 million': 4000000,
    '4,500': 4500,
    '4.1 million': 4100000,
    '4.3 million': 4300000,
    '4.35-7.15 million': (4350000 + 7150000) / 2,  # Average of the range
    '4.4 million': 4400000,
    '4.7 million': 4700000,
    '40,074': 40074,
    '400': 400,
    '40–60 million': (40000000 + 60000000) / 2,  # Average of the range
    '41,129': 41129,
    '42 million': 42000000,
    '420,000': 420000,
    '43 million': 43000000,
    '4351752': 4351752,
    '45 million': 45000000,
    '450,000': 450000,
    '460,000': 460000,
    '47.8 million': 47800000,
    '470,000': 470000,
    '48 million': 48000000,
    '489 million': 489000000,
    '5 million': 5000000,
    '5,454,000': 5454000,
    '5.1 million': 5100000,
    '5.2 million': 5200000,
    '5.5 million': 5500000,
    '5.6 million': 5600000,
    '5.7 million': 5700000,
    '5.8 million': 5800000,
    '50 million': 50000000,
    '51 million': 51000000,
    '510,000': 510000,
    '520,000': 520000,
    '53': 53,
    '540,000': 540000,
    '55,000': 55000,
    '55,720': 55720,
    '56 million': 56000000,
    '56,000': 56000,
    '575,900': 575900,
    '58,000': 58000,
    '583,410': 583410,
    '590,000-1,712,000': (590000 + 1712000) / 2,  # Average of the range
    '597,450': 597450,
    '6 million': 6000000,
    '6,000': 6000,
    '6,000,000': 6000000,
    '6,165': 6165,
    '6.0 million': 6000000,
    '6.5 million': 6500000,
    '6.6 million': 6600000,
    '6.7 million': 6700000,
    '60 million': 60000000,
    '60,000': 60000,
    '600,000': 600000,
    '623,500': 623500,
    '63,000': 63000,
    '67 million': 67000000,
    '68,260': 68260,
    '69.0 million': 69000000,
    '6900': 6900,
    '7 million': 7000000,
    '7,162,000': 7162000,
    '7,200,000': 7200000,
    '7.2 million': 7200000,
    '7.6 million': 7600000,
    '70 million': 70000000,
    '700,000': 700000,
    '72,000': 72000,
    '75 million': 75000000,
    '750,000': 750000,
    '76 million': 76000000,
    '76,000,000': 76000000,
    '76.8 million': 76800000,
    '77 million': 77000000,
    '770,000': 770000,
    '778,000': 778000,
    '790,000': 790000,
    '8 million': 8000000,
    '8.1 million': 8100000,
    '8.2 million': 8200000,
    '8.8 million': 8800000,
    '80 million': 80000000,
    '80,500': 80500,
    '80.4 million': 80400000,
    '800,000': 800000,
    '800,000 – 1 million': 900000,  # Average of the range
    '810,000': 810000,
    '82 million': 82000000,
    '83 million': 83000000,
    '860,000': 860000,
    '86000': 86000,
    '87,000': 87000,
    '8–10 million': (8000000 + 10000000) / 2,  # Average of the range
    '9,600': 9600,
    '9.5 million': 9500000,
    '9.6 million': 9600000,
    '9.8 million': 9800000,
    '9.85 million': 9850000,
    '90–95 million': (90000000 + 95000000) / 2,  # Average of the range
    '96,000': 96000,
    '99,200': 99200,
    '997': 997,
    '999685': 999685,
    '< 500,000': 500000,
    'c. 12 million': 12000000,
    'c. 3.5 million': 3500000,
    'c. 32 million': 32000000,
    'c. 700,000': 700000,
    'nan': np.nan,
    '~10 million': 10000000,
    '~128 million': 128000000,
    '~24,000': 24000,
    '±500000': 500000,
    '≈ 900,000': 900000
}


## Baseline: Dictionaries + Glottolog

In [17]:
langs_with_dicts = pd.read_csv('/Users/go202/Desktop/EQUATE/panlex.csv')
glottolog = pd.read_csv('/Users/go202/Desktop/EQUATE/languages_and_dialects_geo.csv')
merged_data = pd.merge(langs_with_dicts, glottolog, left_on='language', right_on='isocodes', how='inner')
merged_data['dictionary'] = 1
merged_data = merged_data.drop(columns = ['Unnamed: 0', 'language'])

# Language Family and subfamily

In [18]:
newick_file = '/Users/go202/Downloads/tree_glottolog_newick.txt'
trees = Phylo.parse(newick_file, "newick")
glottocode_to_root = {}
# Iterate over all trees
glottocode_pattern = re.compile(r'\[([a-z0-9]+)\]')
for tree_idx, tree in enumerate(trees, start=1):
    root = tree.root  # Get the root node of the current tree
    
    # Extract the root's name
    root_name = root.name if root.name else "Unnamed Root"
    
    # Traverse all clades in the tree to find glottocodes
    for clade_idx, clade in enumerate(tree.find_clades(), start=1):
        if clade.name:
            # Extract the first glottocode using regex
            match = glottocode_pattern.search(clade.name)
            if match:
                glottocode = match.group(1)  # Extract the matched glottocode
                glottocode_to_root[glottocode] = root_name
            else:
                print(f"No glottocode found in clade: {clade.name}")

# Map the root nodes to the DataFrame
merged_data["root_node"] = merged_data["glottocode"].map(glottocode_to_root)

## Bibles - (needs double checking)


In [19]:
bibles = pd.read_csv('/Users/go202/Desktop/EQUATE/bible_languages.csv')
merged_data['bible'] = merged_data['isocodes'].apply(lambda x: 1 if x in bibles['code'].values else 0)
more_langs = [
    "Afrikaans", "Ainu", "Albanian", "Aleut", "Amharic", "Amis", "Apache", "Arabic", 
    "Aramaic", "Armenian", "Assamese", "Auca", "Waodani", "Huaorani", "Australian Kriol", 
    "Avar", "Azerbaijani", "Basque", "Batak", "Belarusian", "Bemba", "Cibemba", "Bengali", 
    "Bhili", "Bisaya-Inunhan", "Blackfoot", "Breton", "Bulgarian", "Burmese", "Buryat", 
    "Cakchiquel", "Carrier", "Catalan", "Caucasian Albanian", "Cheyenne", "Cherokee", 
    "Chichewa", "Nyanja", "Chinese", "Choctaw", "Chope", "Tshopi", "Comanche", "Coptic", 
    "Cornish", "Corsican", "Cree", "Creole", "Croatian", "Czech", "Dakota", "Danish", 
    "Dene Suline", "Chipewyan", "Dutch", "Dzongkha", "English", "Esperanto", "Estonian", 
    "Finnish", "Filipino", "French", "Gagauz", "Geʽez", "Georgian", "German", "Gilbertese", 
    "Gothic", "Greek", "Gujarati", "Gullah", "Gwichʼin", "Haida", "Haitian Creole", "Hakka", 
    "Hawaiian", "Hawaiian Pidgin English", "Hebrew", "Hindi", "Hmong", "Hopi", "Hungarian", 
    "Icelandic", "Igbo", "Ilocano", "Indonesian", "Inupiaq", "Inupiat", "Irish", "Italian", 
    "Jamaican Patois", "Japanese", "Jèrriais", "Kalmyk", "Oirat", "Kannada", "Kashubian", 
    "Kazakh", "Keres", "Khmer", "Khoekhoegowab", "Kikamba", "Kamba", "Konkani", "Korean", 
    "Koryak", "Koyukon", "Kurdish", "Kyrgyz", "Ladakhi", "Lao", "Lacandon", "Lahu", "Latin", 
    "Latvian", "Lisu", "Lithuanian", "LOLcat", "Macedonian", "Malagasy", "Malay", "Malayalam", 
    "Maltese", "Manchu", "Mandarin Chinese", "Manx", "Māori", "Marathi", "Miao", "Micmac", "Mixtec", "Mohawk", 
    "Mongolian", "Naro", "Navajo", "Naxi", "Nepali", "Norwegian", "Nubian", "Nobiin", 
    "Mijikenda", "Nyika", "Occitan", "Odia", "Oriya", "Ojibwa", "O'odham", "Oromo", 
    "Oshindonga", "Otjiherero", "Paniya", "Pashto", "Persian", "Pipil", "Nawat", "Polish", 
    "Portuguese", "Prekmurje", "Quechua", "Runa simi", "Quenya", "Provençal", "Romani", 
    "Romanian", "Romansh", "Russian", "Sakha", "Sanskrit", "Scots", "Scottish Gaelic", 
    "Gàidhlig", "Sechwana", "Seneca", "Serbian", "Seri", "Shawi", "Shan", "Shor", "Sinhala", 
    "Slavonic", "Slovak", "Slovene", "Sogdian", "Sorbian", "Sotho", "Spanish","Standard Arabic", "Swahili", 
    "Swedish", "Syriac", "Tagalog", "Taiwanese", "Tajik", "Tamazight", "Berber", "Tamil", 
    "Tanana", "Upper", "Tashelhayt", "Shilha", "Tatar", "Telugu", "Tewa", "Thai", "Tibetan", 
    "Tlingit", "Tongan", "Tsimshian", "Tulu", "Turkish", "Turkmen", "Tuvan", "Ukrainian", 
    "Urdu", "Uyghur", "Uzbek", "Valencian", "Vietnamese", "Wa", "Wakhi", "Wampanoag", "Welsh", 
    "Xhosa", "Yiddish", "Yoruba", "Yupik", "Zulu", "Zuñi"
]
merged_data.loc[merged_data['name'].isin(more_langs), 'bible'] = 1

## Common Crawl Data 

In [20]:
madlad_data = pd.read_csv('/Users/go202/Desktop/EQUATE/madlad_findings.csv')
madlad_count_map = dict(zip(madlad_data['BCP-47'], madlad_data['clean']))

if 'total*' in madlad_count_map:
    del madlad_count_map['total*']

if 'en*' in madlad_count_map:
    madlad_count_map['en'] = madlad_count_map.pop('en*')
    
madlad_count_map = {iso3_extra_map[iso2]: madlad_count_map[iso2] for iso2 in madlad_count_map if iso2 in iso3_extra_map}
merged_data['cc_GB'] = merged_data['isocodes'].map(madlad_count_map).fillna(0)

## Archival Work

In [21]:
archives = pd.read_csv('/Users/go202/Desktop/EQUATE/all_archives.csv')
archives = archives.groupby('glottocode', as_index=False)['GB'].sum()
archives.rename(columns={'GB': 'archival_GB'}, inplace=True)
merged_data = merged_data.merge(archives[['glottocode', 'archival_GB']], 
                                on='glottocode', 
                                how='left')
merged_data['archival_GB'] = merged_data['archival_GB'].fillna(0)


## HuggingFace Datasets 

In [22]:
hf_data = pd.read_csv('/Users/go202/Desktop/EQUATE/huggingface_work/concat_df.csv')
hf_dataset_number = dict(zip(hf_data['ISO Code'], hf_data['Datasets Number']))
hf_dataset_number =  {iso3_extra_map[iso2]: hf_dataset_number[iso2] for iso2 in hf_dataset_number if iso2 in iso3_extra_map}
merged_data['datasets_n'] = merged_data['isocodes'].map(hf_dataset_number).fillna(0)
hf = pd.read_csv('/Users/go202/Desktop/EQUATE/basic_hf.csv')
hf['iso3'] = hf['ISO Code'].map(iso3_extra_map).fillna(hf['ISO Code'])
hf = hf[['Datasets Count','Models Count', 'iso3']]
# Iterate over retry_hf rows to update the values conditionally
for idx, row in merged_data.iterrows():
    matching_row = hf[hf['iso3'] == row['isocodes']]
    if not matching_row.empty:
        if row['datasets_n'] == 0:
            merged_data.at[idx, 'datasets_n'] = matching_row.iloc[0]['Datasets Count']

## Number of Wikipedia Articles 

In [23]:
wiki = pd.read_csv('/Users/go202/Desktop/EQUATE/basic_wiki_info.csv')
wiki_count_map = dict(zip(wiki['Script (ISO15924 code)'], wiki['WPcode']))
wiki_count_map = {iso3_extra_map[iso2]: wiki_count_map[iso2] for iso2 in wiki_count_map if iso2 in iso3_extra_map}
merged_data['n_wiki_articles'] = merged_data['isocodes'].map(wiki_count_map).fillna(0)

## Number of sentences in opus 

In [24]:
opus = pd.read_csv('/Users/go202/Downloads/all_langs_for_OPUSs.txt', sep=",")
opus['language'] = opus['language'].replace(iso3_extra_map)
opus = opus[['language', 'sentences']]
opus = opus.groupby('language', as_index=False).agg({'sentences': 'sum'})
merged_data = merged_data.merge(opus[['language', 'sentences']], left_on='isocodes', right_on='language', how='left')
merged_data['OPUS'] = merged_data['sentences'].fillna(0)
merged_data = merged_data.drop(columns = ['language', 'sentences'])

## Hours of Speech in Xeus 

In [25]:
xeus = pd.read_excel('/Users/go202/Downloads/xeus_langs_durs.xlsx')
xeus.rename(columns={'Column1': 'name', 'Column2':'XEUS'}, inplace=True)
merged_data['name_lower'] = merged_data['name'].str.lower()
xeus['name_lower'] = xeus['name'].str.lower()
xeus = xeus.groupby('name_lower', as_index=False)['XEUS'].sum()
merged_data = pd.merge(merged_data, xeus, on='name_lower', how='left')
merged_data['XEUS'].fillna(0, inplace=True)

/var/folders/hw/3z11r39j0gd4kgzxbjj6v8z80000gp/T/ipykernel_38307/1081017166.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  merged_data['XEUS'].fillna(0, inplace=True)


## Minutes of Speech in Wikitongues

In [26]:
wikitongues = pd.read_json('/Users/go202/Downloads/Wikitongues/wikitongues.json')
unique_wikitongues = wikitongues[wikitongues['language_list'].apply(lambda x: len(x) == 1)]
unique_wikitongues['name_lower'] = unique_wikitongues['language_list'].apply(lambda x: x[0].lower() if len(x) == 1 else None)
unique_wikitongues.rename(columns={'length_in_minute': 'wikitongues'}, inplace=True)
unique_wikitongues = unique_wikitongues[['wikitongues', 'name_lower']]
unique_wikitongues = unique_wikitongues.groupby('name_lower', as_index=False)['wikitongues'].sum()
merged_data = pd.merge(merged_data, unique_wikitongues, on='name_lower', how='left')
merged_data['wikitongues'].fillna(0, inplace=True)

/var/folders/hw/3z11r39j0gd4kgzxbjj6v8z80000gp/T/ipykernel_38307/2010817563.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  unique_wikitongues['name_lower'] = unique_wikitongues['language_list'].apply(lambda x: x[0].lower() if len(x) == 1 else None)
/var/folders/hw/3z11r39j0gd4kgzxbjj6v8z80000gp/T/ipykernel_38307/2010817563.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  unique_wikitongues.rename(columns={'length_in_minute': 'wikitongues'}, inplace=True)
/var/folders/hw/3z11r39j0gd4kgzxbjj6v8z80000gp/T/ipykernel_38307/2010817563.py:8: FutureWarning: A val

# Models 

## Huggingface Models 

In [27]:
hf_data = pd.read_csv('/Users/go202/Desktop/EQUATE/huggingface_work/concat_df.csv')
hf_model_number = dict(zip(hf_data['ISO Code'], hf_data['Model Number']))
hf_model_number =  {iso3_extra_map[iso2]: hf_model_number[iso2] for iso2 in hf_model_number if iso2 in iso3_extra_map}
merged_data['model_n'] = merged_data['isocodes'].map(hf_model_number).fillna(0)
hf = pd.read_csv('/Users/go202/Desktop/EQUATE/basic_hf.csv')
hf['iso3'] = hf['ISO Code'].map(iso3_extra_map).fillna(hf['ISO Code'])
hf = hf[['Datasets Count','Models Count', 'iso3']]
# Iterate over retry_hf rows to update the values conditionally
for idx, row in merged_data.iterrows():
    matching_row = hf[hf['iso3'] == row['isocodes']]
    if not matching_row.empty:
        if row['model_n'] == 0:
            merged_data.at[idx, 'model_n'] = matching_row.iloc[0]['Models Count']

### Ewe Nawene

In [28]:
merged_data.loc[merged_data['isocodes'] == 'unk', 'model_n'] = 2

# Digital Language Vitality

## Huggingface datasets downloads 

In [129]:
hf_downloads = pd.read_csv('/Users/go202/Desktop/EQUATE/huggingface_work/results/UNFILTERED_huggingface_likes_and_downloads.csv')
hf_downloads['ISO Code'] = hf_downloads['Language'].str.split(':').str[1]
mean_downloads = dict(zip(hf_downloads['ISO Code'], hf_downloads['mean_downloads']))
mean_downloads = {iso3_extra_map[iso2]: mean_downloads[iso2] for iso2 in mean_downloads if iso2 in iso3_extra_map}
merged_data['mean_hf_downloads'] = merged_data['isocodes'].map(mean_downloads).fillna(0)

## Huggingface models downloads

## Timestemp of Huggingface contributions 

In [130]:
hf_data = pd.read_csv('/Users/go202/Desktop/EQUATE/huggingface_work/concat_df.csv')
earliest_huggingface = dict(zip(hf_data['ISO Code'], hf_data['earliest_datetime']))
earliest_huggingface = {iso3_extra_map[iso2]: earliest_huggingface[iso2] for iso2 in earliest_huggingface if iso2 in iso3_extra_map}
merged_data['earliest_datetime'] = merged_data['isocodes'].map(earliest_huggingface).fillna(0)

## Typology of Wikipedia Contributions

In [131]:
wikipedia_active_users = pd.read_json('/Users/go202/Desktop/EQUATE/wikipedia/df.json')
wikipedia_active_users.rename(columns={
        "Column_2": "Language",
        "Column_4": "Script (ISO15924 code)", 
        "Column_6": "Users"
    },
    inplace=True
)
wiki_isos = wiki[['Wikipedia namein native language', 'Script (ISO15924 code)']]
wikipedia_activity = pd.merge(
    wikipedia_active_users,
    wiki_isos[['Wikipedia namein native language', 'Script (ISO15924 code)']],
    left_on='Language',  
    right_on='Wikipedia namein native language',  
    how='left'
)
languages_with_nan_script = wikipedia_activity[wikipedia_activity['Script (ISO15924 code)_x'].isna()]['Language']
languages_with_nan_script_iso = {
    'zho': 'Chinese(writtenvernacular Chinese)',
    'nob': 'Norwegian(Bokmål)',
    'bel': 'Belarusian(officialNarkamaŭkaorthography)',
    'nor': 'Norwegian(Nynorsk)',
    'kmr': 'Kurdish(Kurmanji)',
    'bel': 'Belarusian(Taraškievicaorthography)',
    'ckb': 'Kurdish(Sorani)',
    'quh': 'Quechua(Southern Quechua)',
    'bho': 'Bihari(Bhojpuri)',
    'bod': 'Central Tibetan(Lhasa Tibetan)',
    'kok': 'Konkani(Goan Konkani)',
    'cbk': 'ZamboangaChavacano',
    'zha': 'Zhuang(Standard Zhuang)',
    'bua': 'Buryat(Russia Buriat)',
    'syc': 'Aramaic(Syriac)',
    'dtp': 'Kadazandusun',
    'iba': 'Iban language',
    'gur': 'Farefare(Gurene)',
    'rmy': 'Romani(Vlax Romani)',
    'ann': 'Obolo',
    'tdd': 'Tai Nüa',
    'nbl': 'Southern Ndebele'
} 
wikipedia_activity['Script (ISO15924 code)'] = wikipedia_activity.apply(
    lambda row: next((iso_code for iso_code, name in languages_with_nan_script_iso.items() if name == row['Language']), row['Script (ISO15924 code)_x']) 
    if pd.isna(row['Script (ISO15924 code)_x']) else row['Script (ISO15924 code)_x'],
    axis=1
)
wikipedia_activity['Users'] = wikipedia_activity['Users'].replace({r'\s+': '', r'[,٬]': '', r'\.': ''}, regex=True)

def is_latin_script(s):
    return bool(re.match(r'^[A-Za-z0-9\s,]+$', s))

def convert_to_int(row):
    if is_latin_script(row):
        cleaned = row.replace(' ', '').replace(',', '').replace('.', '')
        return int(cleaned)  
    return row  

wikipedia_activity['Users'] = wikipedia_activity['Users'].apply(convert_to_int)
#wikipedia_activity['Users'] = wikipedia_activity['Users'].replace('一〇五六三五', 105635)

#might want to check if there is any string of non-latin character left
'''
non_integer_values = wikipedia_activity['Users'].apply(lambda x: False if pd.isna(x) else str(x).strip().isdigit()) == False
print(wikipedia_activity.loc[non_integer_values, 'Users'])
'''

wiki_activity_count_map = dict(zip(wikipedia_activity['Script (ISO15924 code)'], wikipedia_activity['Users']))
wiki_activity_count_map = {iso3_extra_map[iso2]: wiki_activity_count_map[iso2] for iso2 in wiki_activity_count_map if iso2 in iso3_extra_map}

def numeral_to_int(numeral_string):
    return int(''.join(str(unicodedata.digit(char)) for char in numeral_string))

wiki_activity_count_map = {
    k: numeral_to_int(v) if isinstance(v, str) else v 
    for k, v in wiki_activity_count_map.items()
}


merged_data['wiki_active_users'] = merged_data['isocodes'].map(wiki_activity_count_map).fillna(0)

# Linguistic Resources 

## ACL anthology papers 

## Language Institutional Status

In [132]:
merged_data['institutional_status'] = 0
institutionality_data_ethnologue = pd.read_csv('/Users/go202/Downloads/language data - ethnologue languages(1).csv')
institush = dict(zip(institutionality_data_ethnologue['Name'], institutionality_data_ethnologue['ethnolog_Vitality']))
merged_data['institutional_status'] = merged_data['name'].map(institush).fillna(0)
merged_data['institutional_status'] = np.where(merged_data['institutional_status'] == 'institutional', 1, 0)

## Language Endangerment status 

In [133]:
endang_status = pd.read_csv('/Users/go202/Desktop/EQUATE/all_unesco.csv')
unesco_endangerment_map = dict(zip(endang_status['Glottocode'], endang_status['Endangerment Status']))
merged_data['endangerment_status'] = merged_data['glottocode'].map(unesco_endangerment_map).fillna(0)

## Typological Features

uriel_distance_&_heuristics
'EQUATE/complete_matrix_g.csv'

# Language Socio Economics

## Geographical Location of Speakers 

In [134]:
merged_data['geometry'] = merged_data.apply(lambda row: Point(row['longitude'], row['latitude']), axis=1)
gdf = gpd.GeoDataFrame(merged_data, geometry='geometry', crs="EPSG:4326")

# Load world boundaries shapefile using geopandas
world = gpd.read_file('/Users/go202/Downloads/ne_10m_admin_0_countries.zip')
#world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))

# Perform a spatial join to add country names based on latitude and longitude
merged_data = gpd.sjoin(gdf, world[['geometry', 'ADMIN']], how='left', predicate='within')
merged_data = merged_data.drop(columns = ['index_right'])

### Subnational Geography 

In [135]:
GDL_shapefiles = gpd.read_file('/Users/go202/Downloads/GDL Shapefiles V6.3.zip')
polygons = dict(zip(GDL_shapefiles['gdlcode'], GDL_shapefiles['geometry']))
centroids_gdf = gpd.GeoDataFrame(
    merged_data,
    geometry=gpd.points_from_xy(merged_data['longitude'], merged_data['latitude']),
    crs=GDL_shapefiles.crs  # Make sure the CRS matches the regions
)
merged_data = gpd.sjoin(centroids_gdf, GDL_shapefiles, how="left", predicate="within")
merged_data = merged_data.drop(columns = ['index_right','continent', 'geometry'])


#run just if necessary 
#merged_data['admin_1_polygon'] = merged_data['gdlcode'].map(polygons).fillna(0)

merged_data = merged_data.dropna(axis=1, how='all')
merged_data.rename(columns={'gdlcode': 'admin_1_code', 'ADMIN': 'country'}, inplace=True)

## Number of Speakers 

WIKI+ETHNOLOGUE+UNESCO+https://www.languagecourse.net/languages-worldwide?starts-with=a

In [136]:
langs = pd.read_csv('/Users/go202/Downloads/language data - ethnologue languages(2).csv')
n_speakers_refined = pd.read_csv('/Users/go202/Desktop/languages_data.csv')

In [137]:
speakers_counts = dict(zip(endang_status['Glottocode'], endang_status['Speakers']))
merged_data['n_speakers'] = merged_data['glottocode'].map(speakers_counts).fillna(0)

def speakers_to_numeric(value):
    if isinstance(value, (int, float)):
        return value
    if 'to' in value:
        range_parts = value.replace('.', '').split(' to ')
        numeric_values = [int(part) for part in range_parts]
        return sum(numeric_values) / len(numeric_values)  # Return average
    else:
        return int(value.replace(',', '').replace('.', ''))

langs = pd.read_csv('/Users/go202/Downloads/language data - ethnologue languages(2).csv')
langs['L1 speakers'] = langs['L1 speakers'].replace(n_speakers_map)
langs = langs.dropna(subset = ['L1 speakers'])

merged_data['n_speakers'] = merged_data['n_speakers'].apply(speakers_to_numeric)
merged_data = merged_data.merge(langs[['Name', 'L1 speakers']], 
                                 left_on='name', 
                                 right_on='Name', 
                                 how='left')
merged_data['n_speakers'] = merged_data['L1 speakers'].combine_first(merged_data['n_speakers'])
merged_data = merged_data.drop(columns=['Name', 'L1 speakers'])

In [138]:
refined_speakers = merged_data.merge(n_speakers_refined, left_on='isocodes', right_on='ISO Alpha-3', how='left')
refined_speakers['Speakers'] = pd.to_numeric(refined_speakers['Speakers'], errors='coerce')
refined_speakers['n_speakers'] = pd.to_numeric(refined_speakers['n_speakers'], errors='coerce')

refined_speakers['n_speakers'] = refined_speakers.apply(
    lambda row: row['Speakers'] if row['Speakers'] < row['n_speakers'] else row['n_speakers'],
    axis=1)

merged_data['n_speakers'] = refined_speakers['n_speakers']

## Universities 

In [139]:
universities_data = pd.read_csv('/Users/go202/Desktop/EQUATE/universities/all_universities_locations.csv')
universities_data = universities_data.dropna(subset=['latitude', 'longitude'], how='all')
geometry = [Point(xy) for xy in zip(universities_data['longitude'], universities_data['latitude'])]
gdf = gpd.GeoDataFrame(universities_data, geometry=geometry)
gdf.set_crs("EPSG:4326", allow_override=True, inplace=True)
centroids_gdf = gpd.GeoDataFrame(
    merged_data,
    geometry=gpd.points_from_xy(merged_data['longitude'], merged_data['latitude']),
    crs=GDL_shapefiles.crs  # Make sure the CRS matches the regions
)

# Only get valid geometries
def valid_point(geom):
    return geom is not None and isinstance(geom, Point) and not geom.is_empty and np.isfinite(geom.x) and np.isfinite(geom.y)

# Filter valid geometries
gdf_languages = centroids_gdf[centroids_gdf.geometry.apply(valid_point)].copy()
gdf_universities = gdf[gdf.geometry.apply(valid_point)].copy()

# Extract coordinates as numpy arrays for building the cKDTree
language_coords = np.array(list(gdf_languages.geometry.apply(lambda geom: (geom.x, geom.y))))
university_coords = np.array(list(gdf_universities.geometry.apply(lambda geom: (geom.x, geom.y))))

# Build the KDTree and query distances
university_tree = cKDTree(university_coords)
distances, indices = university_tree.query(language_coords, k=2)

# Add nearest university data to the dataframe
gdf_languages['nearest_uni'] = [gdf_universities.iloc[idx]['University Name'] if idx < len(gdf_universities) else None for idx in indices[:, 0]]
gdf_languages['second_nearest_uni'] = [gdf_universities.iloc[idx]['University Name'] if idx < len(gdf_universities) else None for idx in indices[:, 1]]
distances_km = distances / 1000
gdf_languages['distance_to_nearest_km'] = distances_km[:, 0]
gdf_languages['distance_to_second_nearest_km'] = distances_km[:, 1]

In [140]:
merged_data = gdf_languages

## Urbanity Rank

In [141]:
zipfile = "/Users/go202/Downloads/World_Urban_Areas.zip"
states = gpd.read_file(zipfile)
points_gdf = states.to_crs(merged_data.crs)
points_with_polygon = gpd.sjoin(merged_data, points_gdf[['RANK', 'Name', 'geometry']], how='left', predicate='within')
points_with_polygon['RANK'] = points_with_polygon['RANK'].fillna(5.0)
points_with_polygon = points_with_polygon.drop(columns=['index_right'])
points_with_polygon['Name'] = points_with_polygon['Name'].fillna('rural')
points_with_polygon.rename(columns={'RANK': 'urbanity_rank'}, inplace=True)

## HDI

In [142]:
subnat_hdi = pd.read_csv('/Users/go202/Downloads/GDL-Subnational-HDI-data.csv')
hdi = dict(zip(subnat_hdi['GDLCODE'], subnat_hdi['2021']))
merged_data['hdi'] = merged_data['admin_1_code'].map(hdi).fillna(0)

# GDP

In [143]:
ds = xr.open_dataset('/Users/go202/Downloads/GDP_per_capita_PPP_1990_2015_v2(2).nc')
gdp_data = ds['GDP_per_capita_PPP']
gdp_data = gdp_data.to_dataframe().reset_index()
gdp_data = gdp_data.dropna(subset=['GDP_per_capita_PPP'])
sorted_gdp = gdp_data.sort_values(by=['longitude', 'latitude', 'time'], ascending=[True, True, False])
most_recent_gdp = sorted_gdp.groupby(['longitude', 'latitude'], as_index=False).first()
gdf = gpd.GeoDataFrame(most_recent_gdp, 
                       geometry=gpd.points_from_xy(most_recent_gdp.longitude, most_recent_gdp.latitude))
gdf = gdf.set_crs('EPSG:4326')
geo_gdp = gpd.sjoin(gdf, GDL_shapefiles, how="inner", predicate="within")
gdp = dict(zip(geo_gdp['gdlcode'], geo_gdp['GDP_per_capita_PPP']))
merged_data['gdp'] = merged_data['admin_1_code'].map(gdp).fillna(0)

# ITU

In [145]:
itu_internet_availability_metrics = pd.read_csv('/Users/go202/Downloads/itu_data_extract_1736949546391.csv')
itu_security_policy = pd.read_csv('/Users/go202/Downloads/itu_data_extract_1736949686810.zip')

In [146]:
def get_most_recent_data(df, series_name):
    """
    Function to get the most recent data for a given series name.

    Parameters:
    - df: The original DataFrame
    - series_name: The name of the series to filter by (e.g., 'Households with a computer')

    Returns:
    - A DataFrame with the most recent data for the given series name
    """
    # Step 1: Filter for the specific series name
    df_filtered = df[df['seriesName'] == series_name]
    
    # Step 2: Get the most recent year for each country
    most_recent_year = df_filtered.groupby('entityIso')['dataYear'].max().reset_index()

    # Step 3: Merge the result back with the original filtered DataFrame to get the most recent values
    df_most_recent = pd.merge(df_filtered, most_recent_year, on=['entityIso', 'dataYear'])

    return df_most_recent


## ICT policy exists

In [147]:
ict_policy = get_most_recent_data(itu_security_policy, 'Cybersecurity legislation/regulation exist')
ict_policy = dict(zip(ict_policy['entityIso'], ict_policy['dataValue']))
merged_data['Cybersecurity legislation/regulation exist'] = merged_data['iso_code'].map(ict_policy).fillna(0)
merged_data['Cybersecurity legislation/regulation exist'] = merged_data['Cybersecurity legislation/regulation exist'].map({'Yes': 1, 'No': 0, 0: 0})

## Cybersecurity legislation/regulation exist

In [148]:
cybersec = get_most_recent_data(itu_security_policy, 'Cybersecurity legislation/regulation exist')
cybersec = dict(zip(cybersec['entityIso'], cybersec['dataValue']))
merged_data['Cybersecurity legislation/regulation exist'] = merged_data['iso_code'].map(cybersec).fillna(0)
merged_data['Cybersecurity legislation/regulation exist'] = merged_data['Cybersecurity legislation/regulation exist'].map({'Yes': 1, 'No': 0, 0: 0})

# Households with computers - Incomplete, needs interpolation

In [149]:
households_with_computer = get_most_recent_data(itu_internet_availability_metrics, 'Households with a computer')
merged_data = pd.merge(merged_data, households_with_computer[['entityIso', 'dataValue']], 
                     left_on='iso_code', right_on='entityIso', how='left')

# Remove the redundant 'entityIso' column (optional)
merged_data.drop(columns=['entityIso'], inplace=True)
merged_data.rename(columns={'dataValue': 'households with computer'}, inplace=True)
iso_codes_with_nan = merged_data.loc[merged_data['households with computer'].isna(), 'iso_code']
iso_codes_with_nan_set = set(iso_codes_with_nan)
print(iso_codes_with_nan_set)

{'CHN', 'JEY', 'PNG', 'GUY', 'LBN', 'FRA', 'TZA', 'XKO', 'NOR', None, 'BEN', 'Z06', 'ISL', 'GNB', 'WSM', 'ETH', 'TON', 'SYR', 'VUT', 'LAO', 'SSD', 'LUX', 'COD', 'BRB', 'NZL', 'ERI', 'SVK', 'NPL', 'DEU', 'HUN', 'COM', 'TCD', 'SDN', 'PRT', 'SWZ', 'GAB', 'MRT', 'GRL', 'SVN', 'COG', 'PYF', 'VCT', 'IMN', 'TJK', 'SLE', 'LBY', 'NIC', 'LVA', 'GNQ', 'SLB', 'GRC', 'CAF', 'VEN', 'STP', 'GUF', 'TGO', nan, 'GBR', 'GRD', 'REU', 'BGR', 'AFG', 'SWE', 'MYT', 'MMR', 'AUS', 'TLS', 'NAM', 'DMA', 'ROU', 'RWA', 'GTM', 'AUT', 'Z01', 'FSM', 'YEM', 'FJI', 'EST', 'VAT'}


# Phones per capita

In [150]:
phones = get_most_recent_data(itu_internet_availability_metrics, 'Individuals who own a mobile cellular telephone')
merged_data = pd.merge(merged_data, phones[['entityIso', 'dataValue']], 
                     left_on='iso_code', right_on='entityIso', how='left')

# Remove the redundant 'entityIso' column (optional)
merged_data.drop(columns=['entityIso'], inplace=True)
merged_data.rename(columns={'dataValue': 'households with phones'}, inplace=True)

# Households with Internet access at home 

In [151]:
internet = get_most_recent_data(itu_internet_availability_metrics, 'Households with Internet access at home')

merged_data = pd.merge(merged_data, internet[['entityIso', 'dataValue']], 
                     left_on='iso_code', right_on='entityIso', how='left')
merged_data.drop(columns=['entityIso'], inplace=True)
merged_data.rename(columns={'dataValue': 'households with home internet'}, inplace=True)

# Individuals using the internet 

In [152]:
inds_using_internet = pd.read_excel('/Users/go202/Downloads/prop_individuals_using_internet.xlsx')

In [153]:
mapping = inds_using_internet.merge(
    internet,
    left_on="Reference Area Name",
    right_on="entityName",
    how="inner"
)[["Reference Area Name", "entityIso"]]

mapping = internet.set_index("entityName")["entityIso"]

inds_using_internet["Reference Area Name"] = inds_using_internet["Reference Area Name"].map(mapping)
filtered_by_sex_and_area = inds_using_internet[
    (inds_using_internet['Sex'] == 'T') & 
    (inds_using_internet['Reference Area Name'].notna())
]

# Sort by 'Reference Area Name' and 'Time Detail' (descending for most recent dates)
sorted_by_area_and_time = filtered_by_sex_and_area.sort_values(
    by=['Reference Area Name', 'Time Detail'], 
    ascending=[True, False]
)

# Keep only the most recent entry for each 'Reference Area Name'
most_recent_entries = sorted_by_area_and_time.drop_duplicates(
    subset=['Reference Area Name'], 
    keep='first'
)

merged_data = merged_data.merge(
    most_recent_entries[['Reference Area Name', 'Observation Value']],
    left_on='iso_code',
    right_on='Reference Area Name',
    how='left'
)

# Rename the 'Observation Value' column to 'individuals_using_internet'
merged_data.rename(columns={'Observation Value': 'individuals_using_internet'}, inplace=True)

# Drop the 'Reference Area Name' column from the merged dataframe (optional)
merged_data.drop(columns=['Reference Area Name'], inplace=True)

# Internet speed per country

In [154]:
data = {
    "Country": [
        "Singapore", "Hong Kong (SAR)", "United Arab Emirates", "France", "Chile", "United States",
        "Iceland", "Denmark", "Spain", "Switzerland", "Romania", "Thailand", "Macau (SAR)", "China",
        "Taiwan", "Israel", "Canada", "Hungary", "Japan", "Portugal", "Netherlands", "Peru", "Kuwait",
        "Liechtenstein", "Brazil", "Qatar", "South Korea", "Poland", "New Zealand", "Luxembourg", "Sweden",
        "Panama", "Colombia", "Jordan", "Vietnam", "Uruguay", "Norway", "Ireland", "Finland", "Malta", "Moldova",
        "United Kingdom", "Trinidad and Tobago", "Saudi Arabia", "Malaysia", "Belgium", "Costa Rica", "Ecuador",
        "Austria", "Slovenia", "Latvia", "Guyana", "Montenegro", "San Marino", "Cyprus", "Paraguay", "Germany",
        "Philippines", "Argentina", "Dominica", "Grenada", "Slovakia", "Italy", "Saint Vincent and the Grenadines",
        "Serbia", "Russia", "Estonia", "Croatia", "Bahrain", "Bulgaria", "Mexico", "Ukraine", "Jamaica", "Uzbekistan",
        "Oman", "Kosovo", "Albania", "Belarus", "Australia", "Egypt", "Czechia", "Venezuela", "Brunei", "Nicaragua",
        "El Salvador", "Mongolia", "Kyrgyzstan", "Kazakhstan", "The Bahamas", "Nepal", "Azerbaijan", "Palestine",
        "India", "Honduras", "Armenia", "Greece", "Guatemala", "Côte d'Ivoire", "Mauritius", "Bangladesh", "Ghana",
        "South Africa", "Turkey", "North Macedonia", "Belize", "Burkina Faso", "Bolivia", "Cambodia", "Dominican Republic",
        "Haiti", "Georgia", "Gabon", "DR Congo", "Laos", "Antigua and Barbuda", "Morocco", "Bosnia and Herzegovina",
        "Iraq", "Tajikistan", "Togo", "Indonesia", "Rwanda", "Fiji", "Myanmar (Burma)", "Madagascar", "Angola", "Uganda",
        "Benin", "Mauritania", "Nigeria", "Sri Lanka", "Zimbabwe", "Tanzania", "Somalia", "Djibouti", "Suriname", "Zambia",
        "Maldives", "Iran", "Algeria", "Pakistan", "Mozambique", "Namibia", "Kenya", "Lebanon", "Tunisia", "Libya", "Yemen",
        "Ethiopia", "Cameroon", "Afghanistan", "Turkmenistan", "Syria", "Cuba"
    ],
    "Mbps": [
        330.98, 306.58, 304.24, 279.22, 276.29, 267.61, 265.25, 240.59, 239.65, 236.98, 236.02, 235.86,
        231.66, 230.11, 224.67, 223.57, 222.28, 206.09, 201.90, 201.87, 200.17, 195.90, 192.99, 188.74,
        185.78, 185.08, 183.20, 183.06, 176.81, 171.17, 167.26, 165.87, 163.75, 162.74, 159.32, 157.94,
        148.78, 145.69, 140.69, 137.15, 134.52, 127.72, 120.98, 120.41, 118.63, 110.01, 104.19, 102.57,
        99.02, 94.96, 94.78, 94.54, 94.40, 94.17, 94.11, 94.10, 94.07, 93.76, 92.86, 92.23, 91.92, 91.45,
        90.68, 90.41, 90.12, 89.70, 89.13, 88.81, 86.61, 86.13, 84.37, 84.17, 82.73, 80.84, 80.36, 80.12,
        79.43, 78.51, 78.47, 77.89, 77.54, 77.10, 76.60, 74.44, 74.24, 74.07, 73.78, 73.67, 71.97, 70.56,
        65.59, 65.17, 62.62, 59.88, 58.58, 57.90, 56.29, 56.10, 51.63, 49.55, 49.33, 48.51, 48.01, 47.96,
        47.84, 47.73, 47.69, 46.14, 45.02, 43.49, 42.07, 40.43, 40.11, 40.06, 39.68, 35.57, 33.67, 33.58,
        33.31, 32.48, 32.07, 31.88, 30.58, 28.94, 27.75, 23.68, 23.52, 23.33, 23.16, 23.02, 22.89, 21.10,
        18.70, 18.53, 18.15, 18.03, 16.35, 16.15, 15.95, 15.65, 15.54, 15.41, 14.66, 14.61, 13.11, 11.72,
        10.44, 10.12, 9.46, 9.32, 3.69, 3.51, 3.38, 2.96
    ]
}

internet_speed_per_country = pd.DataFrame(data)
internet_speed_per_country['iso_code'] = internet_speed_per_country['Country'].map(mapping)


In [155]:
nan_iso_codes = internet_speed_per_country[internet_speed_per_country['iso_code'].isna()]
print(nan_iso_codes)

                Country    Mbps iso_code
1       Hong Kong (SAR)  306.58      NaN
12          Macau (SAR)  231.66      NaN
14               Taiwan  224.67      NaN
26          South Korea  183.20      NaN
34              Vietnam  159.32      NaN
65               Russia   89.70      NaN
75               Kosovo   80.12      NaN
80              Czechia   77.54      NaN
81            Venezuela   77.10      NaN
82               Brunei   76.60      NaN
88          The Bahamas   71.97      NaN
89                Nepal   70.56      NaN
91            Palestine   65.17      NaN
102              Turkey   48.01      NaN
106             Bolivia   47.69      NaN
108  Dominican Republic   45.02      NaN
112            DR Congo   40.11      NaN
113                Laos   40.06      NaN
123     Myanmar (Burma)   28.94      NaN
138                Iran   15.95      NaN
146               Libya   10.44      NaN
152               Syria    3.38      NaN


In [156]:
iso_to_speed = dict(zip(internet_speed_per_country['iso_code'], internet_speed_per_country['Mbps']))

# Add the 'speed' column to merged_data by mapping iso_code
merged_data['speed'] = merged_data['iso_code'].map(iso_to_speed)

# Subnational Internet Speeds 

In [157]:
okla = pd.read_csv('/Users/go202/Desktop/very_important_subnational_speed.csv')

In [158]:
okla['average_tests_devices'] = (okla['tests'] + okla['devices']) / 2

# Deduplicate by keeping the row with the highest average_tests_devices for each gdlcode
okla_deduplicated = okla.loc[okla.groupby('gdlcode')['average_tests_devices'].idxmax()]

# Drop the temporary column if no longer needed
okla_deduplicated.drop(columns=['average_tests_devices'], inplace=True)


In [159]:
merged_data = pd.merge(
    merged_data,
    okla_deduplicated[['gdlcode', 'avg_d_kbps', 'avg_u_kbps', 'avg_lat_ms']],
    left_on='admin_1_code',
    right_on='gdlcode',
    how='left'
)

merged_data.drop(columns=['gdlcode'], inplace=True)

# Human Capital - needs manual correction

In [160]:
stem_graduates = pd.read_excel('/Users/go202/Downloads/P_US STEM (ISCED and Tertiary).xlsx')

In [161]:
stem_graduates.replace("..", np.nan, inplace=True)

year_columns = sorted(
    [col for col in stem_graduates.columns if col[0].isdigit()],
    key=lambda x: int(x.split(" ")[0])  # Sort by year ascending
)

def get_most_recent_value(row):
    # Iterate over year columns from left to right (oldest to newest)
    valid_values = row[year_columns].dropna()  # Remove NaNs
    return valid_values.iloc[-1] if not valid_values.empty else None  # Take the last valid value

# Apply the function across all rows
stem_graduates["Most_Recent_Value"] = stem_graduates.apply(get_most_recent_value, axis=1)

/var/folders/hw/3z11r39j0gd4kgzxbjj6v8z80000gp/T/ipykernel_1323/3206996677.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  stem_graduates.replace("..", np.nan, inplace=True)


In [162]:
stem_graduates_number = dict(zip(stem_graduates['Country Code'], stem_graduates['Most_Recent_Value']))
merged_data['% Stem Grads'] = merged_data['iso_code'].map(stem_graduates_number).fillna(0)

# Literacy rates

In [163]:
country_wise_literacy = pd.read_csv('/Users/go202/Downloads/API_SE.ADT.LITR.ZS_DS2_en_csv_v2_160/API_SE.ADT.LITR.ZS_DS2_en_csv_v2_160.csv',skiprows=4)

In [164]:
year_columns = sorted(
    [col for col in country_wise_literacy.columns if col[0].isdigit()],
    key=lambda x: int(x.split(" ")[0])  # Sort by year ascending
)
country_wise_literacy["Most_Recent_Value"] = country_wise_literacy.apply(get_most_recent_value, axis=1)
country_wise_literacy_number = dict(zip(country_wise_literacy['Country Code'], country_wise_literacy['Most_Recent_Value']))
merged_data['literacy rate'] = merged_data['iso_code'].map(country_wise_literacy_number).fillna(100)

# Educational Attainment

In [165]:
ed_attainment = pd.read_csv('/Users/go202/Downloads/GDL-Educational-index--data.csv')

In [166]:
ed_attainment_value = dict(zip(ed_attainment['GDLCODE'], ed_attainment['2021']))
merged_data['educational_attainment_score'] = merged_data['admin_1_code'].map(ed_attainment_value).fillna(0)

# R&D expenditure

In [167]:
rnd_expenditure = pd.read_csv('/Users/go202/Downloads/API_GB.XPD.RSDV.GD.ZS_DS2_en_csv_v2_77.csv', skiprows=4)

In [168]:
year_columns = sorted(
    [col for col in rnd_expenditure.columns if col[0].isdigit()],
    key=lambda x: int(x.split(" ")[0])  # Sort by year ascending
)
rnd_expenditure["Most_Recent_Value"] = rnd_expenditure.apply(get_most_recent_value, axis=1)
rnd_expenditure_number = dict(zip(rnd_expenditure['Country Code'], rnd_expenditure['Most_Recent_Value']))
merged_data['RnD % of GDP'] = merged_data['iso_code'].map(rnd_expenditure_number).fillna(0)

In [169]:
merged_data

,glottocode,name,isocodes,level,macroarea,latitude,longitude,dictionary,root_node,bible,...,households with home internet,individuals_using_internet,speed,avg_d_kbps,avg_u_kbps,avg_lat_ms,% Stem Grads,literacy rate,educational_attainment_score,RnD % of GDP
0,ghot1243,Ghotuo,aaa,language,Africa,7.115510,5.956630,1,Atlantic-Congo [atla1278],0,...,34.6000,35.456265,23.02,113638.0,21016.0,28.0,0.00000,63.155540,0.706,0.28444
1,alum1246,Arum,aab,language,Africa,9.015130,8.559560,1,Atlantic-Congo [atla1278],0,...,34.6000,35.456265,23.02,89943.0,9571.0,37.0,0.00000,63.155540,0.513,0.28444
2,arii1243,Ari,aac,language,Papunesia,-7.955260,142.400000,1,Suki-Gogodala [suki1244],0,...,27.9065,26.991043,NaN,12590.0,5492.0,647.0,0.00000,70.063683,0.497,0.03158
3,amal1242,Amal,aad,language,Papunesia,-4.045170,141.995000,1,Sepik [sepi1257],0,...,27.9065,26.991043,NaN,206167.0,20800.0,112.0,0.00000,70.063683,0.427,0.03158
4,arbe1236,Arbëreshë Albanian,aae,language,Eurasia,38.911000,16.716400,1,Indo-European [indo1319],1,...,83.6821,85.060745,90.68,340159.0,184233.0,22.0,24.19916,99.000000,0.778,1.45392
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5998,yong1276,Yongbei Zhuang,zyb,language,Eurasia,23.100000,108.150000,1,Tai-Kadai [taik1256],1,...,80.9409,75.611316,230.11,53005.0,34727.0,158.0,0.00000,97.000000,0.664,2.43260
5999,yang1286,Yang Zhuang,zyg,language,Eurasia,23.091864,106.393188,1,Tai-Kadai [taik1256],0,...,80.9409,75.611316,230.11,53005.0,34727.0,158.0,0.00000,97.000000,0.664,2.43260
6000,youj1238,Youjiang Zhuang,zyj,language,Eurasia,23.750000,106.900000,1,Tai-Kadai [taik1256],0,...,80.9409,75.611316,230.11,53005.0,34727.0,158.0,0.00000,97.000000,0.664,2.43260
6001,yong1275,Yongnan Zhuang,zyn,language,Eurasia,22.800000,108.000000,1,Tai-Kadai [taik1256],0,...,80.9409,75.611316,230.11,53005.0,34727.0,158.0,0.00000,97.000000,0.664,2.43260


In [170]:
merged_data.to_csv('/Users/go202/Desktop/complete_matrix_feb.csv')

# Subnational Electricity Access 

In [171]:
zho_row = merged_data[merged_data['isocodes'] == 'cmn']

# Print the row
print(zho_row)

     glottocode              name isocodes     level macroarea  latitude  \
1066   mand1415  Mandarin Chinese      cmn  language   Eurasia   40.0209   

      longitude  dictionary                root_node  bible  ...  \
1066    116.228           1  Sino-Tibetan [sino1245]      1  ...   

     households with home internet  individuals_using_internet   speed  \
1066                       80.9409                   75.611316  230.11   

     avg_d_kbps  avg_u_kbps avg_lat_ms  % Stem Grads  literacy rate  \
1066   321558.0     72355.0       25.0           0.0           97.0   

      educational_attainment_score  RnD % of GDP  
1066                         0.846        2.4326  

[1 rows x 48 columns]


In [172]:
ara_row = merged_data[merged_data['isocodes'] == 'arb']

# Print the row
print(ara_row)

    glottocode             name isocodes     level macroarea  latitude  \
290   stan1318  Standard Arabic      arb  language   Eurasia   27.9625   

     longitude  dictionary                root_node  bible  ...  \
290    43.8525           1  Afro-Asiatic [afro1255]      1  ...   

    households with home internet  individuals_using_internet   speed  \
290                         100.0                       100.0  120.41   

    avg_d_kbps  avg_u_kbps avg_lat_ms  % Stem Grads  literacy rate  \
290   170965.0     96646.0       16.0      21.13605           98.0   

     educational_attainment_score  RnD % of GDP  
290                         0.799       0.46335  

[1 rows x 48 columns]


In [344]:
trial.to_csv('/Users/go202/Desktop/complete_matrix_w.csv')

In [ ]:
smart_phones = get_most_recent_data(itu_availability, 'Individuals who own a smart telephone') # not good, ignore 

In [155]:
skills = get_most_recent_data(itu_availability, 'Individuals with basic ICT in information and data literacy') #not really good 

In [131]:
skills_df = pd.read_csv('/Users/go202/Downloads/ict_skills.csv')

In [132]:
def calculate_mean_ict_skills(df):
    # Step 1: Identify the most recent dataYear for each country and ICT skill
    most_recent = df.groupby(['entityName', 'seriesName'])['dataYear'].max().reset_index()
    
    # Step 2: Merge to filter the rows corresponding to the most recent year
    filtered_df = pd.merge(df, most_recent, on=['entityName', 'seriesName', 'dataYear'])
    
    # Step 3: Calculate the mean value for each country across all ICT skills
    mean_ict_skills = filtered_df.groupby('entityName')['dataValue'].mean().reset_index()
    mean_ict_skills.rename(columns={'dataValue': 'mean_ict_value'}, inplace=True)
    
    return mean_ict_skills

# Usage
# Replace `your_dataframe` with the actual DataFrame variable
mean_ict_skills_df = calculate_mean_ict_skills(skills_df)

## Global Age Structure 

In [111]:
merged_data

,glottocode,name_left,isocodes,level,macroarea,latitude,longitude,dictionary,bible,cc_GB,...,admin_1_code,iso_code,n_speakers,geometry,nearest_uni,second_nearest_uni,distance_to_nearest_km,distance_to_second_nearest_km,hdi,root_node
0,ghot1243,Ghotuo,aaa,language,Africa,7.115510,5.956630,1,0,0,...,NGAr104,NGA,9999.0,POINT (5.95663 7.11551),"Achievers University, Owo",Ambrose Alli University,0.000371,0.000416,0.627,Atlantic-Congo [atla1278]
1,alum1246,Arum,aab,language,Africa,9.015130,8.559560,1,0,0,...,NGAr135,NGA,9999.0,POINT (8.55956 9.01513),"Federal University, Lafia",PIGIER Benin,0.000505,0.000620,0.575,Atlantic-Congo [atla1278]
2,arii1243,Ari,aac,language,Papunesia,-7.955260,142.400000,1,0,0,...,PNGr101,PNG,99.0,POINT (142.40000 -7.95526),The University of Goroka,Divine Word University,0.003542,0.004355,0.565,Suki-Gogodala [suki1244]
3,amal1242,Amal,aad,language,Papunesia,-4.045170,141.995000,1,0,0,...,PNGr115,PNG,999.0,POINT (141.99500 -4.04517),Universitas Cenderawasih,Universitas Ottow Geissler,0.001960,0.001969,0.532,Sepik [sepi1257]
4,arbe1236,Arbëreshë Albanian,aae,language,Eurasia,38.911000,16.716400,1,1,0,...,ITAr110,ITA,99999.0,POINT (16.71640 38.91100),Università degli Studi Magna Graecia di Catanzaro,Università della Calabria,0.000146,0.000666,0.848,Indo-European [indo1319]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5998,yong1276,Yongbei Zhuang,zyb,language,Eurasia,23.100000,108.150000,1,1,0,...,CHNr120,CHN,9999999.0,POINT (108.15000 23.10000),Guangxi Arts University,Guangxi University for Nationalities,0.000255,0.000272,0.734,Tai-Kadai [taik1256]
5999,yang1286,Yang Zhuang,zyg,language,Eurasia,23.091864,106.393188,1,0,0,...,CHNr120,CHN,999999.0,POINT (106.39319 23.09186),Baise University,Youjiang Medical University for Nationalities,0.000826,0.000829,0.734,Tai-Kadai [taik1256]
6000,youj1238,Youjiang Zhuang,zyj,language,Eurasia,23.750000,106.900000,1,0,0,...,CHNr120,CHN,999999.0,POINT (106.90000 23.75000),Youjiang Medical University for Nationalities,Baise University,0.000316,0.000325,0.734,Tai-Kadai [taik1256]
6001,yong1275,Yongnan Zhuang,zyn,language,Eurasia,22.800000,108.000000,1,0,0,...,CHNr120,CHN,9999999.0,POINT (108.00000 22.80000),Guangxi Arts University,Guangxi University for Nationalities,0.000231,0.000239,0.734,Tai-Kadai [taik1256]


In [112]:
df_university_rankings = pd.read_csv('/Users/go202/Downloads/2024 QS World University Rankings 1.1 (For qs.com).csv')

In [114]:
merged_data['nearest_uni'] = merged_data['nearest_uni'].str.lower()
df_university_rankings['Institution Name'] = df_university_rankings['Institution Name'].str.lower()

# Step 2: Merge the dataframes on the appropriate columns
result_df = pd.merge(merged_data, df_university_rankings[['Institution Name', '2024 RANK', 'RES.']], 
                     left_on='nearest_uni', right_on='Institution Name', how='left')

In [120]:
uff = result_df.sort_values(by = '2024 RANK', ascending = True)

In [122]:
uff.to_csv('/Users/go202/Downloads/see.csv')

In [119]:
pwd

'/Users/go202/Library/CloudStorage/GoogleDrive-go292@csah.cam.ac.uk/.tmp/207/Desktop/EQUATE/December_2024'

In [360]:
age_structure = pd.read_csv('/Users/go202/Desktop/EQUATE/global_age_structure.csv')
merged_data = pd.merge(merged_data, age_structure, left_on ='country', right_on='Country', how = 'left')
merged_data = merged_data.drop(columns=['Country', 'Unnamed: 0'])

KeyError: 'country'

In [362]:
pwd

'/Users/go202/Desktop/EQUATE/December_2024'

In [40]:
merged_data.to_csv('complete_matrix_f.csv')